In [49]:
import pandas as pd
import requests

from IPython.display import display, Markdown


# Open-Meteo weather-code descriptions
weather_descriptions = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Heavy drizzle",
    56: "Light freezing drizzle",
    57: "Heavy freezing drizzle",
    61: "Light rain",
    63: "Moderate rain",
    65: "Heavy rain",
    66: "Light freezing rain",
    67: "Heavy freezing rain",
    71: "Light snow",
    73: "Moderate snow",
    75: "Heavy snow",
    77: "Snow grains",
    80: "Light rain showers",
    81: "Moderate rain showers",
    82: "Heavy rain showers",
    85: "Light snow showers",
    86: "Heavy snow showers",
    95: "Thunderstorm",
    96: "Thunderstorm with light hail",
    99: "Thunderstorm with heavy hail"
}

def style_table(dataframe, caption):
    return (dataframe.style.hide(axis="index")
        .set_caption(caption)
        .set_table_styles([{"selector": "caption", "props": [
            ("caption-side", "top"),
            ("font-size", "18px"),
            ("font-weight", "bold"),
            ("color", "#17365d"),
            ("text-align", "left"),
            ("padding", "8px")]}]))


# Ask the user for request parameters
airport = input("Enter airport IATA code (example: LAX): ").strip().upper()

report_date = input("Enter date between 2019-01-01 and 2023-08-31 (YYYY-MM-DD): ").strip()

url = "http://127.0.0.1:8001/api/airport/status"

response = requests.get(
    url,
    params={
        "airport": airport,
        "date": report_date
    },
)


if response.status_code == 200:
    data = response.json()

    weather = data["weather"]
    flights = data["flight_status"]
    travelers = data["traveler_information"]
    holiday = data["holiday_information"]

    weather_code = weather["weather_code"]

    # Airport heading
    display(Markdown(f"""## {data["airport"]} ({data["iata_code"]}) - **{data["municipality"]}, {data["region"]} | {data["date"]}**"""))

    # Weather DataFrame
    weather_df = pd.DataFrame({
        "Weather Metric": ["Conditions",
                            "Weather Code",
                            "Mean Temperature",
                            "Mean Cloud Cover",
                            "Mean Wind Speed",
                            "Mean Relative Humidity",
                            "Total Precipitation"
                        ],
        
        "Value": [weather_descriptions.get(weather_code),
            weather_code,
            f'{weather["temperature_2m_mean"]:,.1f} °F',
            f'{weather["cloud_cover_mean"]:,.0f}%',
            f'{weather["wind_speed_10m_mean"]:,.1f} mph',
            f'{weather["relative_humidity_2m_mean"]:,.0f}%',
            f'{weather["precipitation_sum"]:,.3f} in']
        })

    # Traveler DataFrame
    traveler_df = pd.DataFrame({
        "Traveler Metric": [
            "Approximate Yearly Passengers",
            "Approximate Yearly Departures",
            "Approximate Yearly Arrivals"
        ],
        "Value": [
            f'{travelers["approx_passengers"]:,.0f}',
            f'{travelers["approx_departures"]:,.0f}',
            f'{travelers["approx_arrivals"]:,.0f}']
        })

    # Holiday DataFrame
    holiday_df = pd.DataFrame({
        "Holiday Metric": [
            "Within 3 Days of a Holiday",
            "Holiday Name / Nearest Holiday"
        ],
        "Value": [
            "Yes" if holiday["is_holiday"] else "No",
            holiday["holiday_name"]]
        })

    # Flight-status DataFrame
    flight_df = pd.DataFrame({
        "Flight Metric": [
            "Total Flights",
            "Average Departure Delay",
            "Average Arrival Delay",
            "Departures Delayed 15+ Minutes",
            "Arrivals Delayed 15+ Minutes",
            "Weather-Delayed Flights",
            "Total Weather-Delay Minutes"
        ],
        "Value": [
            f'{flights["total_flights"]:,}',
            f'{flights["average_departure_delay"]:,.2f} min',
            f'{flights["average_arrival_delay"]:,.2f} min',
            f'{flights["departures_delayed_15_min"]:,}',
            f'{flights["arrivals_delayed_15_min"]:,}',
            f'{flights["weather_delayed_flights"]:,}',
            f'{flights["weather_delay_minutes"]:,} min']
        })

    display(style_table(flight_df, "Flight Status"))
    display(style_table(weather_df, "Weather Conditions"))
    display(style_table(traveler_df, "Traveler Information"))
    display(style_table(holiday_df, "Holiday Information"))
    

else:
    display(Markdown(
        f"### Request failed\n\n"
        f"**Status code:** `{response.status_code}`\n\n"
        f"**Response:** {response.text}"
    ))

Enter airport IATA code (example: LAX):  2024-12-27
Enter date between 2019-01-01 and 2023-08-31 (YYYY-MM-DD):  2024-12-27


### Request failed

**Status code:** `400`

**Response:** {
  "error": "IATA code must contain three letters."
}
